# 用 Python 使用 FinMind 台股資料庫

這份 Notebook 寫給：**會 Python、常用 Pandas 做資料分析，但沒有學過 SQL 或資料庫課程的人**。

你不需要先成為資料庫專家。完成後，你會知道：

- 這個專案在解決什麼問題
- Raw、Processed、Parquet、DuckDB 和 View 分別是什麼
- 如何把台股行情取成 Pandas DataFrame
- 什麼時候使用未還原價或還原權息價
- 19 個 View 各自有什麼用途
- 為什麼選股回測不能只看「目前還在市」的股票
- 如何判斷資料是否最新、是否有缺漏

> 本 Notebook 全程以 `read_only=True` 連線，不會下載新資料，也不會修改資料庫。

## 1. 這個專案在做什麼？

這是一套本機的台股資料管線。它定期從 FinMind API 抓取資料，保存原始版本，整理成適合分析的格式，再提供一個 DuckDB 查詢入口。

```text
FinMind API
    ↓  ingestion：下載
Raw Parquet
    ↓  processing：清洗、改欄名、轉型別、去重
Processed Parquet
    ↓  DuckDB Views：提供一致的查詢介面
Python / Pandas / Jupyter / DBeaver
```

它目前整理五類資料：

| 資料 | 用途 |
|---|---|
| 未還原日線 | 當時市場實際成交的 OHLCV |
| 還原權息日線 | 適合跨除權息比較與長期報酬分析 |
| 股票基本資料 | 代碼、名稱、市場、產業、是否為指數 |
| 產業鏈 | 一檔股票可能對應多個產業及次產業 |
| 交易日曆 | 哪些日期應該有台股交易資料 |

## 2. 先懂幾個基本概念

如果你熟悉 Pandas，可以這樣對照：

| 資料庫概念 | Pandas 類比 | 說明 |
|---|---|---|
| Table / View | DataFrame | 由列和欄組成的資料集合 |
| Row | DataFrame 的一列 | 例如某檔股票在某一天的行情 |
| Column | Series | 例如 `close`、`volume` |
| SQL query | 篩選與整理 DataFrame 的程式 | 描述想取哪些欄、哪些列 |
| DuckDB | 可直接查 Parquet 的分析引擎 | 不必先把所有資料載入記憶體 |
| View | 預先定義好的虛擬資料表 | 使用時才讀取底層 Parquet |

這個專案真正保存大量資料的是 **Processed Parquet**。`finmind.duckdb` 主要保存 View 定義，不是第二份行情副本。

三層資料的責任：

- `raw/`：接近 API 原貌，主要用來追查與重建，不建議分析者直接使用。
- `processed/`：已清洗、去重、統一型別，是正式資料。
- `finmind.duckdb`：替分析者包裝好 View，是最推薦的使用入口。

## 3. 準備執行環境

請選擇已安裝本專案套件的 Python 3.12 kernel。若尚未安裝 Jupyter：

```powershell
cd <repo>\pipelines\finmind
.\.venv\Scripts\python.exe -m pip install -e ".[dev]"
.\.venv\Scripts\python.exe -m pip install jupyter ipykernel
```

至少需要 `duckdb`、`pandas` 和 `pyarrow`。

In [1]:
from pathlib import Path
import datetime as dt

import duckdb
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

# 從目前工作目錄往上尋找 finmind/pyproject.toml。
search_roots = [Path.cwd(), *Path.cwd().parents]
PROJECT_DIR = next((p for p in search_roots if (p / "pyproject.toml").exists()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError("找不到 finmind 專案，請從 finmind/ 或 notebooks/ 啟動 Jupyter。")

REPO_ROOT = PROJECT_DIR.parents[1]
DB_PATH = REPO_ROOT / "db" / "finmind" / "finmind.duckdb"
if not DB_PATH.exists():
    raise FileNotFoundError(f"找不到資料庫：{DB_PATH}")

print(f"專案：{PROJECT_DIR}")
print(f"資料庫：{DB_PATH}")
print(f"DuckDB 版本：{duckdb.__version__}")

專案：<repo>\pipelines\finmind
資料庫：<repo>\db\finmind\finmind.duckdb
DuckDB 版本：1.5.5


## 4. 連接資料庫

`con` 可以想成 Python 與資料庫之間的工作階段。

一定要設定 `read_only=True`。如果 Notebook 或 DBeaver 使用讀寫連線，可能鎖住資料庫，讓每日更新失敗。

In [2]:
con = duckdb.connect(str(DB_PATH), read_only=True)
print("已用唯讀模式連線")

已用唯讀模式連線


## 5. 只需要先認識這一點 SQL

下面的 SQL：

```sql
SELECT date, close
FROM v_daily_prices
WHERE stock_id = '2330'
ORDER BY date DESC
LIMIT 5;
```

可以讀成：

- `SELECT`：我要哪些欄位
- `FROM`：資料從哪個 View 來
- `WHERE`：只保留哪些列
- `ORDER BY`：如何排序
- `LIMIT`：最多拿幾列

`con.execute(...).df()` 會直接把 SQL 結果轉成 Pandas DataFrame。

In [3]:
example = con.execute("""
    SELECT date, close
    FROM v_daily_prices
    WHERE stock_id = '2330'
    ORDER BY date DESC
    LIMIT 5
""").df()

print(type(example))
example

<class 'pandas.core.frame.DataFrame'>


,date,close
0,2026-09-03,"2,390.0000"
1,2026-09-02,"2,385.0000"
2,2026-09-01,"2,440.0000"
3,2026-08-31,"2,405.0000"
4,2026-08-28,"2,420.0000"


## 6. 不熟 SQL 也能用：先準備 Python 函式

下面包裝幾個常用操作。之後只要傳股票代碼和日期，就能取得 DataFrame。函式內使用 `?` 傳參數，避免自己拼接 SQL 字串。

In [4]:
def get_prices(stock_id, start, end, adjusted=False):
    """取得單一證券在日期區間內的日線，回傳 DataFrame。"""
    start = pd.Timestamp(start).date()
    end = pd.Timestamp(end).date()
    if start > end:
        raise ValueError("start 不可晚於 end")

    view = "v_adj_daily_prices" if adjusted else "v_daily_prices"
    return con.execute(f"""
        SELECT date, stock_id, open, high, low, close, spread,
               volume, turnover_value, transactions, source, ingested_at
        FROM {view}
        WHERE stock_id = ?
          AND year BETWEEN ? AND ?
          AND date BETWEEN ? AND ?
        ORDER BY date
    """, [str(stock_id), start.year, end.year, start, end]).df()


def search_securities(keyword, limit=20):
    """用代碼或名稱尋找證券。"""
    pattern = f"%{keyword}%"
    return con.execute("""
        SELECT stock_id, stock_name, industry_category_norm, type, is_index
        FROM v_stock_info_latest
        WHERE stock_id ILIKE ? OR stock_name ILIKE ?
        ORDER BY stock_id
        LIMIT ?
    """, [pattern, pattern, limit]).df()


def latest_market(limit=20):
    """取得最新交易日成交金額最高的證券。"""
    return con.execute("""
        SELECT p.date, p.stock_id, s.stock_name, s.industry_category_norm AS industry,
               p.close, p.volume, p.turnover_value
        FROM v_daily_prices p
        LEFT JOIN v_stock_info_latest s USING (stock_id)
        WHERE p.date = (SELECT date FROM v_last_trading_day)
        ORDER BY p.turnover_value DESC
        LIMIT ?
    """, [limit]).df()

print("函式已準備完成")

函式已準備完成


## 7. 第一步通常是找股票代碼

台股代碼必須使用字串，例如 `'0050'`。若寫成整數 `50`，前導零會消失。可以用名稱或代碼搜尋。

In [5]:
search_securities("台積")

,stock_id,stock_name,industry_category_norm,type,is_index
0,2330,台積電,半導體業,twse,False


## 8. 取得行情 DataFrame

未還原價格代表當時市場實際報價，適合觀察價格、成交量與成交金額。

In [6]:
prices = get_prices(
    stock_id="2330",
    start="2024-01-01",
    end="2024-12-31",
    adjusted=False,
)

print(f"資料形狀：{prices.shape}")
prices.tail(10)

資料形狀：(242, 12)


,date,stock_id,open,high,low,close,spread,volume,turnover_value,transactions,source,ingested_at
232,2024-12-18,2330,"1,075.0000","1,090.0000","1,070.0000","1,085.0000",10.0000,47331164,51170937776,40248,ticker,2026-08-30 20:47:42.411132+08:00
233,2024-12-19,2330,"1,065.0000","1,070.0000","1,055.0000","1,070.0000",-15.0000,46126529,49061396399,62146,ticker,2026-08-30 20:47:42.411132+08:00
234,2024-12-20,2330,"1,060.0000","1,060.0000","1,035.0000","1,035.0000",-35.0000,56392109,58923043065,85875,ticker,2026-08-30 20:47:42.411132+08:00
235,2024-12-23,2330,"1,065.0000","1,080.0000","1,060.0000","1,080.0000",45.0000,32800319,35200907996,39942,ticker,2026-08-30 20:47:42.411132+08:00
236,2024-12-24,2330,"1,095.0000","1,095.0000","1,080.0000","1,080.0000",0.0000,30069757,32689957470,36423,ticker,2026-08-30 20:47:42.411132+08:00
237,2024-12-25,2330,"1,090.0000","1,090.0000","1,080.0000","1,085.0000",5.0000,16186220,17591123835,28032,ticker,2026-08-30 20:47:42.411132+08:00
238,2024-12-26,2330,"1,085.0000","1,090.0000","1,085.0000","1,085.0000",0.0000,15017636,16328335510,23380,ticker,2026-08-30 20:47:42.411132+08:00
239,2024-12-27,2330,"1,080.0000","1,095.0000","1,080.0000","1,090.0000",5.0000,26151993,28531251520,40184,ticker,2026-08-30 20:47:42.411132+08:00
240,2024-12-30,2330,"1,090.0000","1,095.0000","1,085.0000","1,090.0000",0.0000,27437455,29944447716,28373,ticker,2026-08-30 20:47:42.411132+08:00
241,2024-12-31,2330,"1,080.0000","1,085.0000","1,075.0000","1,075.0000",-15.0000,29239004,31511634720,32969,ticker,2026-08-30 20:47:42.411132+08:00


### 行情欄位說明

| 欄位 | 意義 |
|---|---|
| `date` | 交易日期 |
| `stock_id` | 股票、ETF 或指數代碼 |
| `open/high/low/close` | 開盤、最高、最低、收盤價 |
| `spread` | 漲跌價差 |
| `volume` | 成交量 |
| `turnover_value` | 成交金額 |
| `transactions` | 成交筆數 |
| `source` | `ticker` 或 `market`，表示資料抓取方式 |
| `ingested_at` | 這筆 processed 資料寫入的時間 |

拿到的就是一般 DataFrame，可以照平常方式分析。

In [7]:
analysis = prices.assign(
    daily_return=prices["close"].pct_change(),
    ma20=prices["close"].rolling(20).mean(),
    volume_ma20=prices["volume"].rolling(20).mean(),
)

analysis[["date", "close", "daily_return", "ma20", "volume", "volume_ma20"]].tail(10)

,date,close,daily_return,ma20,volume,volume_ma20
232,2024-12-18,"1,085.0000",0.0093,"1,047.3000",47331164,"37,031,249.8000"
233,2024-12-19,"1,070.0000",-0.0138,"1,050.3000",46126529,"37,103,577.2000"
234,2024-12-20,"1,035.0000",-0.0327,"1,050.0500",56392109,"38,297,710.0000"
235,2024-12-23,"1,080.0000",0.0435,"1,052.5500",32800319,"37,215,530.3500"
236,2024-12-24,"1,080.0000",0.0000,"1,056.0500",30069757,"37,153,551.6000"
237,2024-12-25,"1,085.0000",0.0046,"1,060.3000",16186220,"36,232,755.4500"
238,2024-12-26,"1,085.0000",0.0000,"1,064.3000",15017636,"35,125,334.8000"
239,2024-12-27,"1,090.0000",0.0046,"1,069.0000",26151993,"34,251,868.0000"
240,2024-12-30,"1,090.0000",0.0000,"1,071.7500",27437455,"33,957,069.6000"
241,2024-12-31,"1,075.0000",-0.0138,"1,072.7500",29239004,"32,987,415.6000"


## 9. 何時使用還原權息價？

除息或拆股會造成報價跳動，但不一定代表投資人真的遭受相同比例的損失。

- 想看當時真正成交價格：使用 `adjusted=False`。
- 想研究單一標的的長期走勢、跨除權息比較：通常使用 `adjusted=True`。

還原權息歷史可能因新的公司行動而整段改變，因此專案每週完整刷新。

> **做選股回測要特別注意。** 還原權息資料只涵蓋 3,149 檔證券；已下市的 767 檔裡只有 7 檔有還原價。這是 FinMind 上游的缺口，不是本專案漏抓（已逐檔向 API 驗證）。
>
> 所以你必須二選一：
>
> - 要**沒有存活者偏誤**的 universe → 用未還原價（`adjusted=False`），自行處理除權息。
> - 要**現成的還原價** → 用 `adjusted=True`，但那是「只剩存活者」的市場，結論要標註這個限制。
>
> 單一標的的長期報酬不受影響。受影響的是「從一籃子股票裡挑選」這類研究。


In [8]:
adj_prices = get_prices("0050", "2003-07-01", "2003-07-31", adjusted=True)
raw_prices = get_prices("0050", "2003-07-01", "2003-07-31", adjusted=False)

comparison = raw_prices[["date", "close"]].merge(
    adj_prices[["date", "close"]].rename(columns={"close": "adj_close"}),
    on="date",
    how="left",
)
comparison["adj_factor"] = comparison["adj_close"] / comparison["close"]
comparison.head()

,date,close,adj_close,adj_factor
0,2003-07-01,38.0500,4.5419,0.1194
1,2003-07-02,38.6900,4.6183,0.1194
2,2003-07-03,39.0000,4.6553,0.1194
3,2003-07-04,39.2600,4.6864,0.1194
4,2003-07-07,41.0000,4.8941,0.1194


## 10. 全部 View 的功能

View 可以當作唯讀 DataFrame 使用。選 View 時，最重要的是先問：**每一列代表什麼？** 這稱為資料粒度（grain）。不同粒度的資料直接合併，可能造成列數意外增加。

### A. 價格 Views

| View | 每列代表 | 何時使用 |
|---|---|---|
| `v_daily_prices` | 一檔證券的一個交易日 | 一般 OHLCV、量價、當時真實報價 |
| `v_adj_daily_prices` | 一檔證券的一個交易日 | 長期走勢、還原權息報酬與回測 |
| `v_prices_combined` | 一檔證券的一個交易日 | 同列比較 `close`、`adj_close`、`adj_factor` |
| `v_daily_prices_enriched` | 行情加上最新股票名稱與分類 | 探索分析；大型查詢優先用原始價格 View |

行情 View 的唯一識別通常是 `(stock_id, date)`。

In [9]:
# DESCRIBE 類似查看 DataFrame 的 dtypes。替換 View 名稱即可探索欄位。
con.execute("DESCRIBE v_daily_prices").df()

,column_name,column_type,null,key,default,extra
0,date,DATE,YES,None,None,None
1,stock_id,VARCHAR,YES,None,None,None
2,open,DOUBLE,YES,None,None,None
3,high,DOUBLE,YES,None,None,None
4,low,DOUBLE,YES,None,None,None
5,close,DOUBLE,YES,None,None,None
6,spread,DOUBLE,YES,None,None,None
7,volume,BIGINT,YES,None,None,None
8,turnover_value,BIGINT,YES,None,None,None
9,transactions,BIGINT,YES,None,None,None


In [10]:
con.execute("DESCRIBE v_adj_daily_prices").df()

,column_name,column_type,null,key,default,extra
0,date,DATE,YES,None,None,None
1,stock_id,VARCHAR,YES,None,None,None
2,open,DOUBLE,YES,None,None,None
3,high,DOUBLE,YES,None,None,None
4,low,DOUBLE,YES,None,None,None
5,close,DOUBLE,YES,None,None,None
6,spread,DOUBLE,YES,None,None,None
7,volume,BIGINT,YES,None,None,None
8,turnover_value,BIGINT,YES,None,None,None
9,transactions,BIGINT,YES,None,None,None


In [11]:
con.execute("DESCRIBE v_prices_combined").df()

,column_name,column_type,null,key,default,extra
0,date,DATE,YES,None,None,None
1,stock_id,VARCHAR,YES,None,None,None
2,open,DOUBLE,YES,None,None,None
3,high,DOUBLE,YES,None,None,None
4,low,DOUBLE,YES,None,None,None
5,close,DOUBLE,YES,None,None,None
6,adj_open,DOUBLE,YES,None,None,None
7,adj_high,DOUBLE,YES,None,None,None
8,adj_low,DOUBLE,YES,None,None,None
9,adj_close,DOUBLE,YES,None,None,None


In [12]:
con.execute("DESCRIBE v_daily_prices_enriched").df()

,column_name,column_type,null,key,default,extra
0,date,DATE,YES,None,None,None
1,stock_id,VARCHAR,YES,None,None,None
2,open,DOUBLE,YES,None,None,None
3,high,DOUBLE,YES,None,None,None
4,low,DOUBLE,YES,None,None,None
5,close,DOUBLE,YES,None,None,None
6,spread,DOUBLE,YES,None,None,None
7,volume,BIGINT,YES,None,None,None
8,turnover_value,BIGINT,YES,None,None,None
9,transactions,BIGINT,YES,None,None,None


### B. 股票基本資料 Views

| View | 每列代表 | 何時使用 |
|---|---|---|
| `v_stock_info_history` | 某代碼在某次歷史分類狀態 | 研究名稱或分類如何改變；同代碼可能多列 |
| `v_stock_info_latest` | 每個代碼目前最新狀態 | 取得名稱、產業、市場類型，最常使用 |
| `v_securities` | 一個目前可交易的非指數代碼 | 建立當下的股票／ETF universe |
| `v_securities_all` | 每個曾經存在的代碼（含已下市） | **回測的 universe**，見下方說明 |
| `v_delisted_securities` | 一個已下市代碼 | 查哪些公司下市、何時下市 |
| `v_indices` | 一個指數代碼 | 查詢加權指數、櫃買或產業指數 |

`stock_id` 不全是數字，也可能是 `0050`、`00679B` 或 `TAIEX`，所以永遠當字串。

**`v_stock_info_latest` 只含目前還在市的證券。** 拿它去 JOIN 歷史行情，會靜默丟掉每一檔後來下市的公司。`v_securities_all` 則是它加上已下市證券，並多出 `is_delisted` 與 `delisted_date` 兩個欄位。

**`v_stock_info_history` 的 `snapshot_date` 是「這組內容第一次被看到的日期」，不是抓取日。** FinMind 每次重抓都會把大多數列的日期改成當天，所以 processed 層是依內容去重、日期取最小值。因此一列帶著某個日期，代表那天有新上市或真正的分類變動。

同一檔股票出現多列有兩種原因，長得很像但意義不同：

- **日期不同** → 分類或名稱在那個時間點被改過（時間上的變動）。
- **日期相同** → 它同時掛在多個分類下，例如 `2330` 同時是 `電子工業` 與 `半導體業`。

想看「目前是什麼」請一律用 `v_stock_info_latest`，它已經每個代碼只留一列。


In [13]:
stock_info_df = con.execute("""
    SELECT *
    FROM v_stock_info_history
    ORDER BY stock_id
""").df()

In [14]:
stock_info_df

,snapshot_date,stock_id,stock_name,industry_category,industry_category_norm,type,is_index
0,2026-08-26,00400A,主動國泰動能高息,ETF,ETF,twse,False
1,2026-08-26,00401A,主動摩根台灣鑫收,ETF,ETF,twse,False
2,2026-08-26,00402A,主動安聯美國科技,ETF,ETF,twse,False
3,2026-08-26,00403A,主動統一升級50,ETF,ETF,twse,False
4,2026-08-26,00404A,主動聯博動能50,ETF,ETF,twse,False
...,...,...,...,...,...,...,...
4316,NaT,TAIEX,加權指數,大盤,大盤,twse,True
4317,NaT,TPEx,櫃買指數,大盤,大盤,tpex,True
4318,NaT,Textiles,紡織纖維類指數,Index,Index,twse,True
4319,NaT,Tourism,觀光事業類指數,Index,Index,twse,True


In [15]:
for i in stock_info_df['industry_category'].unique():
    print(i)

ETF
上櫃ETF
上櫃指數股票型基金(ETF)
受益證券
ETN
指數投資證券(ETN)
水泥工業
其他
食品工業
電器電纜
農業科技業
觀光事業
觀光餐旅
生技醫療業
塑膠工業
建材營造
汽車工業
電子零組件業
綠能環保
紡織纖維
貿易百貨
運動休閒
電子工業
鋼鐵工業
電機機械
電腦及週邊設備業
運動休閒類
半導體業
化學生技醫療
化學工業
其他電子類
玻璃陶瓷
造紙工業
居家生活
橡膠工業
航運業
居家生活類
創新板股票
其他電子業
創新版股票
通信網路業
光電業
電子通路業
資訊服務業
油電燃氣業
數位雲端類
金融保險
文化創意業
綠能環保類
電子商務業
數位雲端
金融業
農業科技
所有證券
存託憑證
Index
大盤


In [16]:
for i in stock_info_df['industry_category_norm'].unique():
    print(i)

ETF
受益證券
ETN
水泥工業
其他
食品工業
電器電纜
農業科技
觀光事業
觀光餐旅
生技醫療業
塑膠工業
建材營造
汽車工業
電子零組件業
綠能環保
紡織纖維
貿易百貨
運動休閒
電子工業
鋼鐵工業
電機機械
電腦及週邊設備業
半導體業
化學生技醫療
化學工業
其他電子業
玻璃陶瓷
造紙工業
居家生活
橡膠工業
航運業
創新板股票
通信網路業
光電業
電子通路業
資訊服務業
油電燃氣業
數位雲端
金融保險
文化創意業
電子商務業
所有證券
存託憑證
Index
大盤


In [17]:
stock_info_df[stock_info_df['industry_category_norm']=='ETF']

,snapshot_date,stock_id,stock_name,industry_category,industry_category_norm,type,is_index
0,2026-08-26,00400A,主動國泰動能高息,ETF,ETF,twse,False
1,2026-08-26,00401A,主動摩根台灣鑫收,ETF,ETF,twse,False
2,2026-08-26,00402A,主動安聯美國科技,ETF,ETF,twse,False
3,2026-08-26,00403A,主動統一升級50,ETF,ETF,twse,False
4,2026-08-26,00404A,主動聯博動能50,ETF,ETF,twse,False
...,...,...,...,...,...,...,...
515,2026-08-26,00995A,主動中信台灣卓越,ETF,ETF,twse,False
516,2026-08-26,00996A,主動兆豐台灣豐收,ETF,ETF,twse,False
517,2026-08-26,00997A,主動群益美國增長,ETF,ETF,twse,False
518,2026-08-26,00998A,主動復華金融股息,上櫃ETF,ETF,tpex,False


In [18]:
con.execute("""
    SELECT stock_id, stock_name, industry_category_norm, type, is_index
    FROM v_stock_info_latest
    WHERE stock_id IN ('0050', '2330', 'TAIEX')
    ORDER BY stock_id
""").df()

,stock_id,stock_name,industry_category_norm,type,is_index
0,0050,元大台灣50,ETF,twse,False
1,2330,台積電,半導體業,twse,False
2,TAIEX,加權指數,大盤,twse,True


In [19]:
# v_securities_all = 目前在市 + 已下市。這是回測應該使用的 universe。
con.execute("""
    SELECT
        count(*)                                 AS 全部,
        count(*) FILTER (WHERE NOT is_delisted)  AS 目前在市,
        count(*) FILTER (WHERE is_delisted)      AS 已下市
    FROM v_securities_all
""").df()


,全部,目前在市,已下市
0,3914,3147,767


In [20]:
# 已下市證券。名稱與下市日期只有部分有值——上游沒有提供其餘的。
con.execute("""
    SELECT stock_id, stock_name, delisted_date, first_seen, last_seen, source
    FROM v_delisted_securities
    WHERE stock_name IS NOT NULL
    ORDER BY delisted_date DESC
    LIMIT 10
""").df()


,stock_id,stock_name,delisted_date,first_seen,last_seen,source
0,1333,恩得利工業股份有限公司,2020-11-17,2005-08-16,2020-03-16,both
1,4947,昂寶電子股份有限公司,2020-05-28,2011-11-15,2020-05-15,both
2,8287,英格爾科技股份有限公司,2020-05-11,2005-01-17,2020-04-16,both
3,3562,頂晶科技股份有限公司,2020-02-27,2007-07-16,2020-02-14,both
4,8913,全銓租賃股份有限公司,2020-01-20,2004-02-19,2020-01-13,both
5,3452,益通光能科技股份有限公司,2020-01-13,2006-03-16,2019-12-16,both
6,4415,台原藥股份有限公司,2019-12-16,2004-02-19,2019-11-15,both
7,1787,福盈科技化學股份有限公司,2019-11-29,2005-05-17,2019-11-15,both
8,5317,凱美電機股份有限公司,2019-09-30,2004-02-19,2019-09-16,both
9,3431,長天科技股份有限公司,2019-09-02,2006-08-16,2019-01-16,both


同一個歷史日期、同一個查詢，只差在 JOIN 哪一個 View。差額就是存活者偏誤的大小：

In [21]:
con.execute("""
    SELECT
        (SELECT count(*) FROM v_daily_prices p
           JOIN v_stock_info_latest s USING (stock_id)
          WHERE p.date = DATE '2010-09-15' AND NOT s.is_index)  AS 只看存活者,
        (SELECT count(*) FROM v_daily_prices p
           JOIN v_securities_all s USING (stock_id)
          WHERE p.date = DATE '2010-09-15' AND NOT s.is_index)  AS 完整universe
""").df()


,只看存活者,完整universe
0,1440,1585


### C. 產業鏈 Views

| View | 每列代表 | 何時使用 |
|---|---|---|
| `v_industry` | 一檔股票與一組產業／次產業關係 | 詳細產業鏈分析；一檔可能有很多列 |
| `v_industry_by_stock` | 一檔股票 | 產業存成 list，適合安全地和股票資料 1:1 合併 |

警告：`v_industry` 是多對多資料。直接和每日行情用 `stock_id` 合併，會把同一天行情複製多次。一般分析優先使用 `v_industry_by_stock`。

In [22]:
con.execute("""
    SELECT stock_id, industries, sub_industries, last_seen
    FROM v_industry_by_stock
    WHERE stock_id = '2330'
""").df()

,stock_id,industries,sub_industries,last_seen
0,2330,[半導體],[晶圓製造],2026-08-29


### D. 交易日曆與資料涵蓋 Views

| View | 每列代表 | 何時使用 |
|---|---|---|
| `v_trading_calendar` | 一個交易日 | 產生預期交易日期；包含未來已公告日期 |
| `v_last_trading_day` | 一列，最新實際有行情的日期 | 判斷資料更新到哪一天 |
| `v_daily_coverage` | 一個已有行情的交易日 | 查看每天有幾筆／幾檔資料與最後寫入時間 |
| `v_known_empty_days` | 一個已確認休市的特殊日期 | 官方日曆說開市、但 API 確認沒有交易的日期 |
| `v_missing_trading_days` | 一個應有但缺少行情的日期 | 找資料缺口；每日／每週工作會嘗試修復 |
| `v_thin_trading_days` | 一個資料量異常少的日期 | 找可能只寫入部分市場資料的日期 |

不要用 `max(date)` 查 `v_trading_calendar` 判斷行情最新日期，因為日曆可能已延伸到年底。請使用 `v_last_trading_day`。

In [23]:
health = con.execute("""
    SELECT
      (SELECT date FROM v_last_trading_day) AS latest_day,
      (SELECT count(*) FROM v_missing_trading_days) AS missing_days,
      (SELECT count(*) FROM v_thin_trading_days) AS thin_days,
      (SELECT count(*) FROM v_known_empty_days) AS known_empty_days
""").df()
health

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,latest_day,missing_days,thin_days,known_empty_days
0,2026-09-03,0,0,1


### E. 執行紀錄 View

| View | 每列代表 | 何時使用 |
|---|---|---|
| `v_ingestion_runs` | 一次 pipeline 工作 | 查看每日更新、每週刷新或 backfill 是否成功、耗時與錯誤數 |

它的資料來自 `db/finmind/logs/runs/*.jsonl`。

In [24]:
con.execute("""
    SELECT job, status, started_at, duration_seconds, error_count
    FROM v_ingestion_runs
    ORDER BY started_at DESC
    LIMIT 15
""").df()

,job,status,started_at,duration_seconds,error_count
0,daily_update,success,2026-09-03 07:57:43.254883,23.7190,0
1,backfill,success,2026-08-30 14:25:15.669660,0.8120,0
2,weekly_refresh,success,2026-08-30 14:02:40.482088,3.5780,0
3,weekly_refresh,success,2026-08-30 13:45:55.176967,3.5630,0
4,weekly_refresh,success,2026-08-30 13:44:34.201596,9.3900,0
5,backfill,success,2026-08-30 12:37:18.965281,"2,352.4380",0
6,discover_delisted,success,2026-08-30 12:32:20.376862,233.6250,0
7,daily_update,success,2026-08-26 06:53:33.911813,10.7810,0
8,daily_update,success,2026-08-26 06:52:44.647620,10.7190,0
9,full_rebuild,success,2026-08-26 06:42:02.147790,595.9850,0


## 11. 用 Python 查看 View 目錄

若忘了有哪些 View，可向 DuckDB 的 `information_schema` 詢問。這相當於列出資料庫目錄。

In [25]:
views = con.execute("""
    SELECT table_name AS view_name
    FROM information_schema.tables
    WHERE table_type = 'VIEW'
    ORDER BY table_name
""").df()
print(f"共有 {len(views)} 個 Views")
views

共有 19 個 Views


,view_name
0,v_adj_daily_prices
1,v_daily_coverage
2,v_daily_prices
3,v_daily_prices_enriched
4,v_delisted_securities
5,v_indices
6,v_industry
7,v_industry_by_stock
8,v_ingestion_runs
9,v_known_empty_days


## 12. 日期範圍 Macro

價格資料按 `year/month` 分區。只寫 `date BETWEEN ...` 時，DuckDB 不一定能跳過其他年份的檔案。專案提供三個 Table Macro，自動補上有利於分區裁剪的條件：

| Macro | 用途 |
|---|---|
| `daily_prices_between(lo, hi)` | 取得日期區間內的未還原行情 |
| `adj_daily_prices_between(lo, hi)` | 取得日期區間內的還原行情 |
| `prices_combined_between(lo, hi)` | 取得日期區間內的還原/未還原並排比較 |

Macro 不是另一份資料，而是帶參數的預先定義查詢。前面的 `get_prices()` 也已手動加入年份條件。

`v_prices_combined` 要 JOIN 兩份資料集，只加 `date` 條件只會裁剪未還原那一側，所以它的 Macro 特別值得用：實測一季 1.31 秒降到 0.29 秒。

In [26]:
con.execute("""
    SELECT stock_id, min(date) AS from_date, max(date) AS to_date, count(*) AS bars
    FROM daily_prices_between('2024-01-01', '2024-03-31')
    WHERE stock_id IN ('0050', '2330')
    GROUP BY stock_id
    ORDER BY stock_id
""").df()

,stock_id,from_date,to_date,bars
0,0050,2024-01-02,2024-03-29,56
1,2330,2024-01-02,2024-03-29,56


## 13. 完整實例：最新成交額排行

這裡直接使用前面包裝好的 Python 函式。結果仍是 DataFrame，可以繼續做圖、統計或輸出。

In [27]:
ranking = latest_market(20)
ranking

,date,stock_id,stock_name,industry,close,volume,turnover_value
0,2026-09-03,TAIEX,加權指數,大盤,"45,857.6600",11187583407,989260563594
1,2026-09-03,Electronic,電子類指數,Index,"2,895.8100",3386510042,793593451275
2,2026-09-03,Semiconductor,半導體類指數,Index,"1,532.1300",986168834,328449458733
3,2026-09-03,TPEx,櫃買指數,大盤,395.2500,984356000,241225809000
4,2026-09-03,ElectronicPartsComponents,電子零組件類指數,Index,892.8600,445377327,208639228041
5,2026-09-03,ComputerPeripheralEquipment,電腦及週邊設備類指數,Index,473.6200,369302540,86462970019
6,2026-09-03,Optoelectronic,光電類指數,Index,75.6100,1237944801,80647330081
7,2026-09-03,2408,南亞科,半導體業,477.5000,123832867,62053039830
8,2026-09-03,CommunicationsInternet,通信網路類指數,Index,236.9200,150432375,41415938865
9,2026-09-03,2454,聯發科,半導體業,"4,340.0000",9319994,40663342040


## 14. 匯出成 CSV

以下將前面的 `prices` DataFrame 匯出到 `db/finmind/notebooks-outputs/`。這不會修改正式資料。

In [28]:
OUTPUT_DIR = REPO_ROOT / "db" / "finmind" / "notebooks-outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_path = OUTPUT_DIR / "2330_2024.csv"
prices.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"已匯出：{output_path}")

已匯出：<repo>\db\finmind\notebooks-outputs\2330_2024.csv


## 15. 常見錯誤

1. **把股票代碼寫成數字**：請寫 `'0050'`，不要寫 `50`。
2. **用交易日曆最大日期當最新行情**：改用 `v_last_trading_day`。
3. **單一標的的長期報酬使用未還原價**：跨除權息研究改用 `v_adj_daily_prices`。
4. **選股回測用 `v_stock_info_latest` 當 universe**：它不含已下市證券，績效會被系統性高估。改用 `v_securities_all`，並先讀第 9 節關於還原價的取捨。
5. **直接把 `v_industry` 和行情合併**：它是多對多資料，可能放大列數；優先用 `v_industry_by_stock`。
6. **對已下市證券做產業分組**：它們沒有產業分類，會全部掉進 NULL 群組。先用 `is_delisted` 篩掉。
7. **一次把全部行情載入 Pandas**：先在 DuckDB 用日期、股票條件篩選，再 `.df()`。
8. **開啟讀寫連線**：分析工作應使用 `read_only=True`，否則會擋住每日與每週排程。
9. **手動修改 Raw 或 Processed Parquet**：正式資料應由 pipeline 腳本更新。


## 16. 資料如何更新？

Notebook 只負責分析。更新資料請在 PowerShell 執行：

```powershell
cd <repo>\pipelines\finmind

# 每日：新日期、最近修訂與歷史缺口
.\.venv\Scripts\python.exe scripts\daily_update.py

# 每週：完整刷新還原權息資料並檢查／修復倉儲
.\.venv\Scripts\python.exe scripts\weekly_refresh.py

# 唯讀健康檢查
.\.venv\Scripts\python.exe scripts\verify.py
```

建議排程：

- 週一至週六 18:30：`scripts/daily_update.bat`
- 星期日 02:00：`scripts/weekly_refresh.bat`

每日漏跑通常不用手動補資料，下一次執行會偵測並修復缺口。

還有一支 `scripts/discover_delisted.py`，用來找回已下市證券。它**不需要排程**：
它處理的是 2026-08-26（第一份證券清單快照）之前就下市的公司，那個集合是歷史的、
不會再長大；之後的下市由每日與每週工作自動接住。大約一年跑一次當稽核即可。

每次資料庫架構或資料範圍有變動，都會在 `docs/changelog/` 留下一份紀錄。

## 17. 接下來可以做什麼？

建議依序嘗試：

1. 把 `get_prices()` 的股票改成你熟悉的標的。
2. 使用 `adjusted=True` 計算長期報酬。
3. 從 `v_stock_info_latest` 選出某個產業，再取得多檔行情。
4. 用 `v_daily_coverage` 畫出每日證券數量。
5. 挑一個歷史日期，比較 JOIN `v_stock_info_latest` 與 `v_securities_all` 的股票數差多少。
6. 將分析結果保存到 `notebooks-outputs/`，不要寫回正式資料層。

如果查詢變慢，先確認是否已在 SQL 層限制股票、日期與 `year`，不要先把千萬列資料全部載入 Pandas。

In [29]:
# 使用完畢後關閉連線。
con.close()
print("DuckDB 連線已關閉")

DuckDB 連線已關閉
